# Bengali Long-Form ASR — Demucs + VAD + Whisper Pipeline

**Pipeline:** Audio → Demucs (vocal separation) → VAD (silence removal + gap padding) → Whisper ASR → Post-processing → Repetition removal

Run on Kaggle with **GPU + Internet** enabled.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q demucs
!pip install -q transformers torchaudio librosa jiwer evaluate
print('✅ Dependencies installed')

## Cell 2 — Imports & Configuration

In [ ]:
import os
import gc
import math
import time
import re
import torch
import torchaudio
import numpy as np
import pandas as pd
import unicodedata
import librosa
import warnings
from tqdm.auto import tqdm
from transformers import pipeline

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


class Config:
    TEST_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
    OUTPUT_DIR = "/kaggle/working"
    ASR_MODEL = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"

    # Demucs
    DEMUCS_MODEL = "htdemucs"
    DEMUCS_CHUNK_S = 120       # process 2 min at a time

    # VAD
    VAD_THRESHOLD = 0.4
    VAD_MIN_SPEECH_MS = 250
    VAD_MIN_SILENCE_MS = 150
    GAP_PADDING_MS = 150       # silence gap between segments

    # ASR chunking
    MAX_CHUNK_S = 25.0
    MIN_CHUNK_S = 1.0


config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
print('✅ Config ready')

## Cell 3 — Load Models (Demucs + VAD + Whisper)

In [ ]:
# ── 1. Demucs ─────────────────────────────────────────
from demucs.pretrained import get_model
from demucs.apply import apply_model

print('Loading Demucs …')
demucs_model = get_model(config.DEMUCS_MODEL)
demucs_model.eval()
if torch.cuda.is_available():
    demucs_model = demucs_model.cuda()
demucs_sr = demucs_model.samplerate  # 44100
print(f'✅ Demucs loaded (SR: {demucs_sr})')

# ── 2. Silero VAD ─────────────────────────────────────
print('Loading Silero VAD …')
vad_model, vad_utils = torch.hub.load(
    'snakers4/silero-vad', model='silero_vad', onnx=False
)
get_speech_timestamps = vad_utils[0]
print('✅ VAD loaded')

# ── 3. Whisper ASR ────────────────────────────────────
print(f'Loading {config.ASR_MODEL} …')
asr_pipe = pipeline(
    task='automatic-speech-recognition',
    model=config.ASR_MODEL,
    device=0 if device == 'cuda' else -1,
    torch_dtype=torch.float16,
    model_kwargs={'attn_implementation': 'sdpa'},
)
print('✅ ASR loaded')
print('\n✅ All models ready')

## Cell 4 — Demucs Vocal Separation

Uses `split=True` so Demucs handles its own internal chunking with proper overlap.  
This gives the same quality as single-pass on short clips.

In [ ]:
def separate_vocals(audio_path: str) -> torch.Tensor:
    """
    Run Demucs on full audio file. Returns vocals as mono torch tensor at demucs_sr.
    Uses Demucs internal splitting for memory safety on long files.
    """
    # Load full audio
    audio, orig_sr = torchaudio.load(audio_path)

    # Resample to Demucs SR
    if orig_sr != demucs_sr:
        audio = torchaudio.transforms.Resample(orig_sr, demucs_sr)(audio)

    # Stereo (Demucs requires it)
    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)

    dur = audio.shape[1] / demucs_sr
    print(f'    Demucs input: {dur:.0f}s')

    # Let Demucs handle its own chunking
    with torch.no_grad():
        sources = apply_model(
            demucs_model,
            audio.unsqueeze(0).cuda() if torch.cuda.is_available() else audio.unsqueeze(0),
            device='cuda' if torch.cuda.is_available() else 'cpu',
            split=True,
            overlap=0.25,
            progress=False,
        )

    # sources: [batch, 4, channels, samples] → index 3 = vocals
    vocals = sources[0, 3].cpu().mean(dim=0)  # mono

    del sources
    torch.cuda.empty_cache()

    return vocals


print('✅ Demucs function defined')

## Cell 5 — VAD Silence Removal with Gap Padding

Removes silence from Demucs output, inserts 150ms gaps between segments  
so adjacent speech doesn't smash together.

In [ ]:
def remove_silence_with_vad(vocals: torch.Tensor, sr: int = 16000) -> torch.Tensor:
    """
    Run VAD on vocals, collect speech segments with gap padding between them.
    Input: mono tensor at given sr.
    Returns: cleaned mono tensor at same sr.
    """
    gap_samples = int(config.GAP_PADDING_MS / 1000 * sr)
    gap_silence = torch.zeros(gap_samples)

    timestamps = get_speech_timestamps(
        vocals,
        vad_model,
        sampling_rate=sr,
        threshold=config.VAD_THRESHOLD,
        min_speech_duration_ms=config.VAD_MIN_SPEECH_MS,
        min_silence_duration_ms=config.VAD_MIN_SILENCE_MS,
    )

    if not timestamps:
        return vocals  # no speech found, return as-is

    parts = []
    for i, ts in enumerate(timestamps):
        parts.append(vocals[ts['start']:ts['end']])
        if i < len(timestamps) - 1:
            parts.append(gap_silence)

    clean = torch.cat(parts, dim=0)

    orig_dur = len(vocals) / sr
    clean_dur = len(clean) / sr
    print(f'    VAD: {orig_dur:.0f}s → {clean_dur:.0f}s ({len(timestamps)} segments, {orig_dur - clean_dur:.0f}s silence removed)')

    return clean


print('✅ VAD function defined')

## Cell 6 — VAD-Based Chunking for ASR

In [ ]:
def chunk_for_asr(audio: torch.Tensor, sr: int = 16000) -> list:
    """Split clean audio into ≤25s chunks using VAD boundaries."""
    timestamps = get_speech_timestamps(
        audio, vad_model,
        sampling_rate=sr,
        threshold=config.VAD_THRESHOLD,
        min_speech_duration_ms=config.VAD_MIN_SPEECH_MS,
        min_silence_duration_ms=100,
    )

    if not timestamps:
        return [(0, len(audio))]

    # Merge close segments (gap < 0.5s) up to max_chunk_s
    merged = []
    cur_s, cur_e = timestamps[0]['start'], timestamps[0]['end']
    for ts in timestamps[1:]:
        gap = (ts['start'] - cur_e) / sr
        new_dur = (ts['end'] - cur_s) / sr
        if gap < 0.5 and new_dur <= config.MAX_CHUNK_S:
            cur_e = ts['end']
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = ts['start'], ts['end']
    merged.append((cur_s, cur_e))

    # Absorb tiny segments into neighbours
    cleaned = []
    for s, e in merged:
        dur = (e - s) / sr
        if dur < config.MIN_CHUNK_S and cleaned:
            prev_s, prev_e = cleaned[-1]
            cleaned[-1] = (prev_s, e)
        else:
            cleaned.append((s, e))

    # Hard-split oversized
    final = []
    for s, e in cleaned:
        dur = (e - s) / sr
        if dur <= config.MAX_CHUNK_S:
            final.append((s, e))
        else:
            n = math.ceil(dur / config.MAX_CHUNK_S)
            clen = (e - s) // n
            for j in range(n):
                cs = s + j * clen
                ce = s + (j+1) * clen if j < n-1 else e
                final.append((cs, ce))

    return final


print('✅ Chunking function defined')

## Cell 7 — Bengali Post-Processor

In [ ]:
class BengaliPostProcessor:
    """Post-processing for Bengali ASR output."""

    def __init__(self):
        self.corrections = {
            # Diacritic errors
            'র্া': 'রা',
            '্্': '',
            'ো': 'ো',
            'অ্য': 'এ্য',
            'য়': 'য়',
            # Specific ASR errors
            'প্রাক্ষে প্রাক্ষে': 'প্রকাশে',
            'প্রাক্ষনে প্রাক্ষনে': 'প্রকাশনে',
            'একস্কিজুনে': 'এক্সকিউশনে',
            'বালো': 'ভালো',
            'মানাব': 'মানবে',
            'নাগিন': 'নাগরিক',
            'ছবল': 'ছবির',
            'গল্পোটি': 'গল্পটি',
            'পাব্লিশের্স': 'পাবলিশার্স',
            'প্রাই঵িট': 'প্রাইভেট',
            'লিমিটিট': 'লিমিটেড',
            'সঙ্রক্খিতো': 'সংগ্রহিত',
            'ভিশাচ': 'বিশেষ',
            'বন্তা': 'বিষয়',
            'জশোবন্কের': 'জবাবের',
        }

    def remove_repetitions(self, text):
        words = text.split()
        if len(words) < 3:
            return text
        cleaned = []
        i = 0
        while i < len(words):
            word = words[i]
            j = i + 1
            while j < len(words) and words[j] == word:
                j += 1
            count = j - i
            if count > 2:
                cleaned.extend([word] * min(2, count))
            else:
                cleaned.extend(words[i:j])
            i = j
        return ' '.join(cleaned)

    def apply_corrections(self, text):
        for wrong, right in self.corrections.items():
            if wrong in text:
                text = text.replace(wrong, right)
        return text

    def process(self, text):
        if not text:
            return ''
        text = unicodedata.normalize('NFC', text)
        bengali_punct = '।.,;:!?\'\"()[]{}—–-–―…॰'
        for p in bengali_punct:
            text = text.replace(p, '')
        text = self.remove_repetitions(text)
        text = self.apply_corrections(text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = unicodedata.normalize('NFC', text)
        return text


post_processor = BengaliPostProcessor()
print('✅ Post-processor ready')

## Cell 8 — N-gram Repetition Removal (Final Pass)

In [ ]:
def remove_consecutive_repetitions(text):
    """Remove repeated n-grams (n=1..5) from text. Final cleanup pass."""
    if not isinstance(text, str):
        return text

    text = unicodedata.normalize('NFC', text)
    words = text.split()
    if not words:
        return text

    limit_n = 5
    has_changed = True

    while has_changed:
        has_changed = False
        for n in range(limit_n, 0, -1):
            i = 0
            new_words = []
            while i < len(words):
                chunk = words[i:i+n]
                if (i + n <= len(words)) and (i + 2*n <= len(words)) and (words[i+n:i+2*n] == chunk):
                    new_words.extend(chunk)
                    i += n
                    while (i + n <= len(words)) and (words[i:i+n] == chunk):
                        i += n
                    has_changed = True
                else:
                    new_words.append(words[i])
                    i += 1
            if has_changed:
                words = new_words
                break

    return unicodedata.normalize('NFC', ' '.join(words))


print('✅ Repetition removal defined')

## Cell 9 — Full Transcription Pipeline

**Audio → Demucs → 16kHz → VAD silence removal → VAD chunking → Whisper → Post-process → Repetition removal**

In [ ]:
SR = 16000  # target sample rate for VAD + ASR


def transcribe_file(audio_path: str) -> str:
    """
    Full pipeline for one audio file:
      1. Demucs vocal separation
      2. Resample to 16kHz
      3. VAD silence removal (with gap padding)
      4. VAD-based chunking (≤25s)
      5. Whisper ASR on each chunk
      6. Post-processing + repetition removal
    """
    file_name = os.path.basename(audio_path)
    print(f'  Processing: {file_name}')

    t0 = time.time()

    # ── Step 1: Demucs ────────────────────────────────
    try:
        vocals = separate_vocals(audio_path)
    except Exception as e:
        print(f'    ⚠ Demucs failed ({e}), falling back to raw audio')
        audio, orig_sr = torchaudio.load(audio_path)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0)
        else:
            audio = audio.squeeze(0)
        if orig_sr != SR:
            audio = torchaudio.transforms.Resample(orig_sr, SR)(audio.unsqueeze(0)).squeeze(0)
        clean_audio = audio
        # skip to ASR
        vocals = None

    if vocals is not None:
        # ── Step 2: Resample to 16kHz ─────────────────
        vocals_16k = torchaudio.transforms.Resample(demucs_sr, SR)(vocals.unsqueeze(0)).squeeze(0)
        del vocals

        # ── Step 3: VAD silence removal ───────────────
        clean_audio = remove_silence_with_vad(vocals_16k, SR)
        del vocals_16k

    gc.collect()
    torch.cuda.empty_cache()

    # ── Step 4: VAD chunking ──────────────────────────
    chunks = chunk_for_asr(clean_audio, SR)
    print(f'    ASR: {len(chunks)} chunks')

    # ── Step 5: Whisper ASR ───────────────────────────
    clean_np = clean_audio.numpy()
    transcriptions = []

    for s, e in chunks:
        chunk = clean_np[s:e]

        # Pad if too short
        if len(chunk) < SR:
            chunk = np.pad(chunk, (0, SR - len(chunk)))

        # Truncate safety
        max_samples = int(30.0 * SR)
        if len(chunk) > max_samples:
            chunk = chunk[:max_samples]

        result = asr_pipe(
            {'array': chunk, 'sampling_rate': SR},
            return_timestamps=False,
        )
        text = result.get('text', '').strip()
        text = unicodedata.normalize('NFC', text)
        if text:
            transcriptions.append(text)

    raw_text = ' '.join(transcriptions)

    # ── Step 6: Post-processing ───────────────────────
    processed = post_processor.process(raw_text)
    final = remove_consecutive_repetitions(processed)
    final = unicodedata.normalize('NFC', final)

    elapsed = time.time() - t0
    print(f'    Done in {elapsed/60:.1f} min | {len(final)} chars')

    del clean_audio, clean_np
    gc.collect()
    torch.cuda.empty_cache()

    return final


print('✅ Pipeline function defined')

In [ ]:
from jiwer import wer, cer

# ── Paths ────────────────────────────────────────────
TRAIN_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/audio"
TRAIN_ANNOT_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/annotation"

# ── Pick first train file ────────────────────────────
train_audio_files = sorted([
    f for f in os.listdir(TRAIN_AUDIO_DIR) if f.endswith('.wav')
])

if not train_audio_files:
    print('⚠ No train audio files found. Skipping validation.')
else:
    VAL_FILE = train_audio_files[0]  # change index to try other files
    VAL_AUDIO_PATH = os.path.join(TRAIN_AUDIO_DIR, VAL_FILE)
    VAL_ANNOT_PATH = os.path.join(
        TRAIN_ANNOT_DIR,
        os.path.splitext(VAL_FILE)[0] + '.txt'
    )

    # ── Load ground truth ────────────────────────────
    with open(VAL_ANNOT_PATH, 'r', encoding='utf-8') as f:
        ground_truth = f.read().strip()
    ground_truth = unicodedata.normalize('NFC', ground_truth)

    info = torchaudio.info(VAL_AUDIO_PATH)
    dur = info.num_frames / info.sample_rate

    print('='*70)
    print('VALIDATION RUN')
    print('='*70)
    print(f'File      : {VAL_FILE}')
    print(f'Duration  : {dur:.0f}s ({dur/60:.1f} min)')
    print(f'GT length : {len(ground_truth)} chars / {len(ground_truth.split())} words')
    print()

    # ── Run full pipeline ────────────────────────────
    val_start = time.time()
    predicted = transcribe_file(VAL_AUDIO_PATH)
    val_elapsed = time.time() - val_start

    # ── Calculate metrics ────────────────────────────
    val_wer = wer(ground_truth, predicted)
    val_cer = cer(ground_truth, predicted)

    print(f'\n{"="*70}')
    print('VALIDATION RESULTS')
    print(f'{"="*70}')
    print(f'WER       : {val_wer:.4f} ({val_wer*100:.1f}%)')
    print(f'CER       : {val_cer:.4f} ({val_cer*100:.1f}%)')
    print(f'Time      : {val_elapsed/60:.1f} min')
    print(f'Predicted : {len(predicted)} chars / {len(predicted.split())} words')
    print(f'GT        : {len(ground_truth)} chars / {len(ground_truth.split())} words')

    # ── Show text comparison ─────────────────────────
    print(f'\n{"─"*70}')
    print('GROUND TRUTH (first 500 chars):')
    print(ground_truth[:500])
    print(f'\n{"─"*70}')
    print('PREDICTED (first 500 chars):')
    print(predicted[:500])

    # ── Save full comparison ─────────────────────────
    comp_path = os.path.join(config.OUTPUT_DIR, 'validation_comparison.txt')
    with open(comp_path, 'w', encoding='utf-8') as f:
        f.write(f'File: {VAL_FILE}\n')
        f.write(f'WER: {val_wer:.4f} ({val_wer*100:.1f}%)\n')
        f.write(f'CER: {val_cer:.4f} ({val_cer*100:.1f}%)\n')
        f.write(f'\n{"="*70}\nGROUND TRUTH:\n{"="*70}\n{ground_truth}\n')
        f.write(f'\n{"="*70}\nPREDICTED:\n{"="*70}\n{predicted}\n')
    print(f'\n✅ Full comparison saved → {comp_path}')

    del predicted
    gc.collect()
    torch.cuda.empty_cache()

## Cell 10 — Process All Test Files

In [ ]:
print('='*70)
print('PROCESSING TEST FILES')
print('='*70 + '\n')

test_files = []
if os.path.exists(config.TEST_AUDIO_DIR):
    test_files = sorted([f for f in os.listdir(config.TEST_AUDIO_DIR) if f.endswith('.wav')])

print(f'Test files: {len(test_files)}')

if test_files:
    results = []
    total_start = time.time()

    for i, audio_file in enumerate(test_files):
        audio_path = os.path.join(config.TEST_AUDIO_DIR, audio_file)
        file_id = os.path.splitext(audio_file)[0]

        print(f'\n[{i+1}/{len(test_files)}] {audio_file}')

        try:
            transcript = transcribe_file(audio_path)
        except Exception as e:
            print(f'  ⚠ FAILED: {e}')
            transcript = ''

        # Fallback if empty
        if not transcript:
            transcript = 'এটি একটি স্বয়ংক্রিয় প্রতিলিপি'

        transcript = unicodedata.normalize('NFC', transcript)

        results.append({
            'filename': file_id,
            'transcript': transcript,
        })

        # Checkpoint every 3 files
        if (i + 1) % 3 == 0:
            pd.DataFrame(results).to_csv(f'{config.OUTPUT_DIR}/checkpoint.csv', index=False)
            print(f'  📌 Checkpoint saved ({i+1}/{len(test_files)})')

    total_elapsed = time.time() - total_start

    # Save submission
    submission_df = pd.DataFrame(results)[['filename', 'transcript']]
    submission_path = f'{config.OUTPUT_DIR}/submission.csv'
    submission_df.to_csv(submission_path, index=False)

    print(f'\n{"="*70}')
    print(f'DONE — {len(test_files)} files in {total_elapsed/60:.1f} min')
    print(f'Submission saved → {submission_path}')
    print(f'{"="*70}')
    print(submission_df.head())

else:
    print('No test files found')
    submission_df = pd.DataFrame({
        'filename': [f'test_{i:03d}' for i in range(1, 25)],
        'transcript': ['এটি একটি নমুনা প্রতিলিপি'] * 24,
    })
    submission_df.to_csv(f'{config.OUTPUT_DIR}/submission.csv', index=False)

## Cell 11 — Final N-gram Cleanup on Submission

In [ ]:
input_file = f'{config.OUTPUT_DIR}/submission.csv'
output_file = f'{config.OUTPUT_DIR}/FOTR-submission.csv'

df = pd.read_csv(input_file)
print(f'Loaded {input_file} with shape {df.shape}')

if 'transcript' in df.columns:
    df['transcript'] = df['transcript'].apply(remove_consecutive_repetitions)
    df.to_csv(output_file, index=False)
    print(f'✅ Final submission saved → {output_file}')

    # Verify
    sample = 'প্রেসিডেন্ট সিরিজের প্রেসিডেন্ট সিরিজের প্রেসিডেন্ট সিরিজের'
    print(f'\nTest: \'{sample}\'')
    print(f'  →   \'{remove_consecutive_repetitions(sample)}\'')
else:
    print('Column transcript not found')

## Cell 12 — Pipeline Summary

In [ ]:
print('='*70)
print('PIPELINE SUMMARY')
print('='*70)
print('''
  Audio → Demucs (vocal separation, split=True)
       → Resample 44.1kHz → 16kHz
       → VAD silence removal (150ms gap padding)
       → VAD chunking (≤25s segments)
       → Whisper bengaliAI medium ASR
       → Bengali post-processing (corrections + word dedup)
       → N-gram repetition removal (n=1..5)
       → NFC normalization
       → submission.csv
''')
print('🎉 Done!')